In [ ]:
import numpy as np
from ase.build import surface
from ase.io import read
from ase.geometry import find_mic
from ase.visualize import view

def verificar_simetria_slab(slab, tol=0.7):
    """
    Verifica se as metades superior e inferior do slab são imagens espelhadas/invertidas,
    tratando corretamente o plano central (estilo palíndromo 12321 ou 1221).
    """
    posicoes = slab.get_positions()
    simbolos = slab.get_chemical_symbols()
    cell = slab.get_cell()
    pbc = slab.get_pbc()
    
    if len(posicoes) <= 1:
        return False

    # Centro real da nuvem de átomos no eixo Z
    z_centro = np.mean(posicoes[:, 2])
    
    posicoes_centradas = posicoes.copy()
    posicoes_centradas[:, 2] -= z_centro
    
    # Separar os átomos em metades superior, inferior e plano central
    # Tolerância de 0.05 para capturar átomos que pertençam exatamente ao plano do meio
    indices_superior = np.where(posicoes_centradas[:, 2] > 0.05)[0]
    indices_inferior = np.where(posicoes_centradas[:, 2] < -0.05)[0]
    
    # Se o topo e a base não tiverem a mesma quantidade de átomos, a simetria já quebrou
    if len(indices_superior) != len(indices_inferior):
        return False
        
    if len(indices_superior) == 0:
        return True # Monocamada perfeita
        
    # Gerar as posições simétricas esperadas para a metade inferior (Inversão espacial)
    posicoes_alvo = -posicoes_centradas[indices_superior].copy()
    posicoes_alvo[:, 2] += z_centro  # Retorna para a escala Z original
    
    # Mapeamento ponto a ponto considerando as condições periódicas de contorno (PBC)
    for idx_sup, pos_alvo in zip(indices_superior, posicoes_alvo):
        elemento_sup = simbolos[idx_sup]
        match_encontrado = False
        
        for idx_inf in indices_inferior:
            if simbolos[idx_inf] != elemento_sup:
                continue
                
            vetor_bruto = pos_alvo - posicoes[idx_inf]
            vetor_mic, v_len = find_mic(vetor_bruto, cell, pbc)
            
            if v_len <= tol:
                match_encontrado = True
                break
                
        if not match_encontrado:
            return False
            
    return True


def slab_preview_symmetric(path_bulk_xsf: str, miller, min_layers: int = 2, max_layers: int = 20):
    """
    Gera o slab e agrupa os átomos em camadas ordenando diretamente por suas coordenadas Z,
    evitando os problemas de cutoff rígido em estruturas distorcidas.
    """
    bulk = read(path_bulk_xsf)
    
    # Garante que o bulk está bem empacotado na própria caixa antes de cortar
    bulk.wrap()
    
    # Criar a superfície estendida na direção normal do plano de Miller
    atoms_surface = surface(bulk, miller, 10, periodic=True)
    
    # --- NOVO AGRUPAMENTO DE CAMADAS INTELIGENTE ---
    # Pegamos as coordenadas Z de todos os átomos gerados na superfície
    z_coords = atoms_surface.get_positions()[:, 2]
    
    # Arredondamos as alturas em até 1 casa decimal para agrupar átomos que estão
    # na mesma "folha" ou plano atômico (equivalente a um cutoff adaptativo de ~0.15 Angstroms)
    z_arredondado = np.round(z_coords, 1)
    alturas_unicas = np.unique(z_arredondado)
    # -----------------------------------------------

    slab_encontrado = False
    slab_final = None

    # Loop testando cortes baseados no número de planos atômicos reais (alturas_unicas)
    for layers in range(min_layers, min(max_layers + 1, len(alturas_unicas))):
        # Seleciona a altura limite do corte para esta iteração
        z_limite = alturas_unicas[layers - 1]
        
        # Filtra os átomos que estão abaixo ou na altura limite
        indices_slab = np.where(z_arredondado <= z_limite)[0]
        
        slab_teste = atoms_surface[indices_slab].copy()
        
        # Centraliza o slab adicionando a região de vácuo necessária para o DFT
        slab_teste.center(vacuum=15, axis=2)
        
        # Testa se esse corte específico gerou um palíndromo estável
        if verificar_simetria_slab(slab_teste, tol=0.25):
            slab_final = slab_teste
            slab_encontrado = True
            print(f"✅ Sucesso! Slab simétrico encontrado com {layers} camadas atômicas.")
            break
            
    if not slab_encontrado:
        print(f"⚠️ Não foi encontrado arranjo perfeitamente simétrico.")
        print(f"Mostrando o menor corte possível ({min_layers} camadas) para análise.")
        z_limite = alturas_unicas[min_layers - 1]
        indices_slab = np.where(z_arredondado <= z_limite)[0]
        slab_final = atoms_surface[indices_slab]
        slab_final.center(vacuum=15, axis=2)

    # Aplica as posições escaladas/fracionárias como no seu código original
    slab_final.set_positions(slab_final.get_scaled_positions())
    return slab_final

In [26]:
# --- CONFIGURAÇÃO ---
ARQUIVO_BULK = "teste.XSF"  # Indique o seu arquivo XSF aqui
INDICES_MILLER = (1, 0, 0)                             # Indique o plano de corte aqui

# Gerar o objeto temporário na memória
meu_slab = slab_preview_symmetric(
    path_bulk_xsf=ARQUIVO_BULK,
    miller=INDICES_MILLER,
    min_layers=3,
    max_layers=15,
#    cutoff=0.1
)

# --- VISUALIZAR COM O PRÓPRIO ASE ---
# No Jupyter, isso abrirá a janela do visualizador padrão do ASE (GUI)
view(meu_slab)

✅ Slab simétrico gerado artificialmente por inversão de simetria (8 átomos).


<Popen: returncode: None args: ['/home/lorran/miniconda3/bin/python', '-m', ...>

In [ ]:
import numpy as np
from ase import Atoms
from ase.build import surface
from ase.io import read
from ase.geometry import find_mic
from ase.visualize import view

def verificar_simetria_estrita(slab, tol=0.25):
    """
    Verifica matematicamente se o slab é um palíndromo perfeito (Z -> -Z)
    ou um palíndromo angulado (Inversão espacial em relação ao centro).
    """
    pos = slab.get_positions()
    simbolos = slab.get_chemical_symbols()
    cell = slab.get_cell()
    pbc = [True, True, False] # Slabs são periódicos apenas em X e Y
    
    centro = np.mean(pos, axis=0)
    pos_centrada = pos - centro
    
    # Teste 1: Espelhamento Puro em Z (Palíndromo paralelo)
    espelhamento_ok = True
    for i, p in enumerate(pos_centrada):
        alvo = np.array([p[0], p[1], -p[2]]) + centro
        encontrado = False
        for j, p_outro in enumerate(pos):
            if simbolos[i] == simbolos[j]:
                _, dist = find_mic(alvo - p_outro, cell, pbc)
                if dist < tol:
                    encontrado = True
                    break
        if not encontrado:
            espelhamento_ok = False
            break
            
    if espelhamento_ok:
        return True, "Espelhamento Z (Palíndromo Perfeito)"
        
    # Teste 2: Inversão Total (Palíndromo Angulado)
    inversao_ok = True
    for i, p in enumerate(pos_centrada):
        alvo = -p + centro
        encontrado = False
        for j, p_outro in enumerate(pos):
            if simbolos[i] == simbolos[j]:
                _, dist = find_mic(alvo - p_outro, cell, pbc)
                if dist < tol:
                    encontrado = True
                    break
        if not encontrado:
            inversao_ok = False
            break
            
    if inversao_ok:
        return True, "Inversão Central (Palíndromo Angulado)"
        
    return False, ""

def construir_slab_palindromo(path_bulk_xsf: str, miller, min_camadas=3, max_camadas=15):
    """
    Varre os planos atômicos reais gerados pelo corte e encontra a menor
    espessura que resulte em um slab perfeitamente simétrico.
    """
    # 1. Ler o bulk original e garantir o empacotamento das coordenadas
    bulk = read(path_bulk_xsf)
    bulk.wrap()
    
    # 2. Gerar a superfície base e estender verticalmente (X4) para ter planos de sobra
    surf_base = surface(bulk, miller, layers=1, periodic=True)
    surf_estendida = surf_base * (1, 1, 4)
    
    posicoes = surf_estendida.get_positions()
    z_coords = posicoes[:, 2]
    
    # 3. Mapear os planos atômicos reais ordenando e agrupando o eixo Z
    z_ordenado = np.sort(z_coords)
    planos_z = []
    grupo_atual = [z_ordenado[0]]
    
    for z in z_ordenado[1:]:
        if z - grupo_atual[-1] > 0.25: # Se a distância for maior que 0.25 A, é um novo plano
            planos_z.append(np.max(grupo_atual))
            grupo_atual = [z]
        else:
            grupo_atual.append(z)
    planos_z.append(np.max(grupo_atual))
    
    # 4. Loop de busca inteligente pelo palíndromo natural
    for n in range(min_camadas, min(max_camadas + 1, len(planos_z) + 1)):
        z_corte = planos_z[n - 1]
        indices_slab = np.where(z_coords <= z_corte + 0.05)[0]
        
        slab_teste = surf_estendida[indices_slab].copy()
        slab_teste.set_pbc([True, True, False])
        
        passou, tipo_simetria = verificar_simetria_estrita(slab_teste)
        if passou:
            print(f"✅ Sucesso! Slab simétrico encontrado com {n} planos atômicos.")
            print(f"-> Tipo de simetria detectada: {tipo_simetria}")
            
            # Centraliza adicionando os 15 Angstroms de vácuo para o seu cálculo de DFT
            slab_teste.center(vacuum=15, axis=2)
            return slab_teste
            
    print("❌ Não foi possível encontrar um palíndromo natural neste intervalo. Tente aumentar o max_camadas.")
    return None

# ==============================================================================
# EXECUÇÃO DO SEU TESTE
# ==============================================================================

# Executa o código na memória do Jupyter utilizando o seu arquivo real
meu_slab = construir_slab_palindromo(path_bulk_xsf="teste.XSF", miller=(1, 0, 0), min_camadas=3, max_camadas=40)

# Abre a janela do ASE para você conferir visualmente o resultado
if meu_slab:
    supercelula = meu_slab * (4, 4, 1)
    view(supercelula)

In [66]:
import numpy as np
from ase import Atoms
from ase.build import surface
from ase.io import read
from ase.geometry import find_mic
from ase.visualize import view

def verificar_simetria_universal(slab, tol=0.15):
    """
    Juiz cego: Verifica Espelhamento (Mirror) e Inversão estritamente pela matemática,
    sem saber nada sobre química ou ligações.
    """
    pos = slab.get_positions()
    syms = np.array(slab.get_chemical_symbols())
    cell = slab.get_cell()
    pbc = [True, True, False]
    
    z_min, z_max = np.min(pos[:, 2]), np.max(pos[:, 2])
    z_mid = (z_max + z_min) / 2.0
    
    # --- 1. TESTE DE ESPELHAMENTO (MIRROR) ---
    mirror_ok = True
    for i in range(len(pos)):
        p1 = pos[i]
        el = syms[i]
        
        # Posição onde o reflexo deveria estar
        p_target = np.copy(p1)
        p_target[2] = 2 * z_mid - p1[2] 
        
        match = False
        for j in range(len(pos)):
            if syms[j] == el:
                dp = p_target - pos[j]
                dp[2] = 0.0  # Avaliamos o Z fora da fronteira periódica
                _, d_xy = find_mic(dp, cell, pbc)
                d_z = abs(p_target[2] - pos[j][2])
                
                if d_xy <= tol and d_z <= tol:
                    match = True
                    break
        if not match:
            mirror_ok = False
            break
            
    if mirror_ok:
        return True, "Mirror (Espelhamento Z)"

    # --- 2. TESTE DE INVERSÃO ESPACIAL ---
    # O centro Z é obrigatoriamente o meio do slab. 
    # Para achar o centro XY, pegamos o átomo mais baixo e testamos com os do topo.
    z_bottom_idx = np.argmin(pos[:, 2])
    p_ref = pos[z_bottom_idx]
    el_ref = syms[z_bottom_idx]
    
    inversion_ok = False
    for i in range(len(pos)):
        # Se for o mesmo elemento e estiver no topo oposto
        if syms[i] == el_ref and abs(pos[i][2] - (2*z_mid - p_ref[2])) <= tol:
            center = (p_ref + pos[i]) / 2.0
            
            # Testa este centro contra todos os átomos
            center_valid = True
            for j in range(len(pos)):
                p_orig = pos[j]
                el_orig = syms[j]
                p_inv = 2 * center - p_orig
                
                match_inv = False
                for k in range(len(pos)):
                    if syms[k] == el_orig:
                        dp = p_inv - pos[k]
                        dp[2] = 0.0
                        _, d_xy = find_mic(dp, cell, pbc)
                        d_z = abs(p_inv[2] - pos[k][2])
                        
                        if d_xy <= tol and d_z <= tol:
                            match_inv = True
                            break
                if not match_inv:
                    center_valid = False
                    break
                    
            if center_valid:
                inversion_ok = True
                break
                
    if inversion_ok:
        return True, "Inversion (Centro de Inversão)"

    return False, "Nenhuma"


def construir_slab_generico(path_bulk, miller, min_camadas=3, max_camadas=10, espessura_max_camada=0.9):
    """
    Gera um super-bloco, mapeia as camadas lógicas e desliza uma janela
    até encontrar um corte naturalmente simétrico.
    """
    print(f"Lendo {path_bulk} e procurando slabs simétricos no plano {miller}...")
    bulk = read(path_bulk)
    return bulk
    
    # 1. Cria um "tijolo" estendido para termos material de sobra para cortar
    tijolo = surface(bulk, miller, layers=max_camadas + 5, periodic=True)
    pos_tijolo = tijolo.get_positions()
    
    # 2. Mapeamento Lógico: Agrupa os átomos em "camadas" baseando-se apenas na altura Z
    z_coords = pos_tijolo[:, 2]
    idx_sort = np.argsort(z_coords)
    
    camadas = []
    camada_atual = [idx_sort[0]]
    
    for idx in idx_sort[1:]:
        # Se a distância Z para o átomo anterior for maior que a espessura permitida, é uma nova camada
        if z_coords[idx] - z_coords[camada_atual[-1]] > espessura_max_camada:
            camadas.append(camada_atual)
            camada_atual = [idx]
        else:
            camada_atual.append(idx)
    camadas.append(camada_atual)
    
    # 3. Janela Deslizante: Testa diferentes pontos de início e diferentes espessuras
    # Testamos começar o slab em diferentes alturas para fugir de cortes ruins do ASE
    for start_idx in range(min(4, len(camadas))):
        
        for n_layers in range(min_camadas, max_camadas + 1):
            end_idx = start_idx + n_layers - 1
            if end_idx >= len(camadas):
                continue
                
            # Extrai os índices de todos os átomos dentro da janela de corte
            indices_corte = []
            for k in range(start_idx, end_idx + 1):
                indices_corte.extend(camadas[k])
                
            # Monta o slab candidato
            candidato = tijolo[indices_corte].copy()
            candidato.set_pbc([True, True, False])
            
            # Reposiciona o candidato para Z=0 para padronizar
            pos_cand = candidato.get_positions()
            pos_cand[:, 2] -= np.min(pos_cand[:, 2])
            candidato.set_positions(pos_cand)
            
            # 4. O Veredito
            is_sym, type_sym = verificar_simetria_universal(candidato)
            
            if is_sym:
                candidato.center(vacuum=15, axis=2)
                print(f"✅ Sucesso! Slab encontrado.")
                print(f"   -> Simetria: {type_sym}")
                print(f"   -> Tamanho: {n_layers} camadas lógicas ({len(candidato)} átomos).")
                return candidato
                
    print("❌ Não foi encontrado nenhum corte simétrico para os parâmetros fornecidos.")
    return None

# ==============================================================================
# EXECUÇÃO
# ==============================================================================
# O parâmetro 'espessura_max_camada' dita o que o código considera "uma camada".
# Para o seu SiC, o Si e o C estão separados por alguns décimos de Angstrom.
# Um valor de 0.8 a 1.0 agrupa os dois na mesma base lógica sem misturar com a próxima bicamada.

meu_slab = construir_slab_generico(
    path_bulk="teste.XSF", 
    miller=(1, 0, 0), 
    min_camadas=3, 
    max_camadas=20,
    espessura_max_camada=0.9 
)

if meu_slab:
    supercelula = meu_slab * (2, 2, 1)
    view(supercelula)

Lendo teste.XSF e procurando slabs simétricos no plano (1, 0, 0)...


0.00s - Debugger warning: It seems that frozen modules are being used, which may
0.00s - make the debugger miss breakpoints. Please pass -Xfrozen_modules=off
0.00s - to python to disable frozen modules.
0.00s - Note: Debugging will proceed. Set PYDEVD_DISABLE_FILE_VALIDATION=1 to disable this validation.


In [65]:
import numpy as np
from ase import Atoms
from ase.build import surface
from ase.io import read
from ase.visualize import view

def construir_slab_com_simetria_forcada(path_bulk, miller, n_camadas_metade=3, delta_z=0.8):
    """
    Corta metade de um slab e o espelha matematicamente no eixo Z para FORÇAR 
    a simetria (ideal para zerar momento de dipolo em DFT de materiais dopados/ligas).
    """
    print(f"Lendo {path_bulk}...")
    bulk = read(path_bulk)
    
    # 1. Gera um bloco do cristal ao longo do plano de corte
    tijolo = surface(bulk, miller, layers=n_camadas_metade * 2, periodic=True)
    pos = tijolo.get_positions()
    
    # 2. Mapeia e agrupa os átomos em "camadas lógicas" baseadas na altura Z
    z_coords = pos[:, 2]
    idx_sort = np.argsort(z_coords)
    
    camadas = []
    camada_atual = [idx_sort[0]]
    
    for idx in idx_sort[1:]:
        # Se a diferença de altura for maior que delta_z, pulamos para a próxima camada
        if z_coords[idx] - z_coords[camada_atual[-1]] > delta_z:
            camadas.append(camada_atual)
            camada_atual = [idx]
        else:
            camada_atual.append(idx)
    camadas.append(camada_atual)
    
    # 3. Extrai apenas a metade inferior do slab (que servirá de molde)
    indices_metade = []
    for k in range(min(n_camadas_metade, len(camadas))):
        indices_metade.extend(camadas[k])
        
    metade_slab = tijolo[indices_metade].copy()
    
    # 4. Encontra o plano de espelhamento (a camada mais alta do nosso molde)
    pos_metade = metade_slab.get_positions()
    syms_metade = np.array(metade_slab.get_chemical_symbols())
    z_max = np.max(pos_metade[:, 2])
    
    # 5. Constrói a metade superior refletindo as coordenadas
    novas_posicoes = list(pos_metade)
    novos_simbolos = list(syms_metade)
    
    for i in range(len(pos_metade)):
        p = pos_metade[i]
        # Reflete todos os átomos, exceto os que já estão exatamente no plano central
        if abs(p[2] - z_max) > 0.1:  
            p_refletido = np.copy(p)
            p_refletido[2] = 2 * z_max - p[2]  # Matemática pura do espelho
            novas_posicoes.append(p_refletido)
            novos_simbolos.append(syms_metade[i])
            
    # 6. Monta o slab simétrico final
    slab_simetrico = Atoms(
        symbols=novos_simbolos, 
        positions=novas_posicoes, 
        cell=metade_slab.get_cell(), 
        pbc=[True, True, False]
    )
    
    # Adiciona o vácuo de forma simétrica nas duas pontas
    slab_simetrico.center(vacuum=15, axis=2)
    
    print(f"✅ Slab construído com simetria forçada!")
    print(f"   -> Átomos na metade base: {len(metade_slab)}")
    print(f"   -> Átomos no slab final simétrico: {len(slab_simetrico)}")
    return slab_simetrico

# ==============================================================================
# EXECUÇÃO
# ==============================================================================
# n_camadas_metade dita quantas camadas a "base" terá antes de ser espelhada.
# Exemplo: Se n=3, o script pega as camadas [A, B, C], espelha ao redor da C, 
# gerando um slab final [A, B, C, B, A].

meu_slab = construir_slab_com_simetria_forcada(
    path_bulk="teste.XSF", 
    miller=(1, 0, 0), 
    n_camadas_metade=10, 
    delta_z=0.8
)

if meu_slab:
    view(meu_slab * (2, 2, 1))

Lendo teste.XSF...
✅ Slab construído com simetria forçada!
   -> Átomos na metade base: 20
   -> Átomos no slab final simétrico: 38


0.00s - Debugger warning: It seems that frozen modules are being used, which may
0.00s - make the debugger miss breakpoints. Please pass -Xfrozen_modules=off
0.00s - to python to disable frozen modules.
0.00s - Note: Debugging will proceed. Set PYDEVD_DISABLE_FILE_VALIDATION=1 to disable this validation.


In [1]:
from elasticipy.tensors.elasticity import StiffnessTensor
import matplotlib.pyplot as plt
import plotly 
import numpy as np
import math
import plotly.graph_objects as go

# 1. Criando o tensor (exemplo com simetria cúbica genérica)
# Substitua pelos valores Cij do material que você está estudando

matriz = [
    [ 87.64,   6.99,  11.91, -17.19,    0.0,    0.0],
    [  6.99,  87.64,  11.91,  17.19,    0.0,    0.0],
    [ 11.91,  11.91, 107.20,    0.0,    0.0,    0.0],
    [-17.19,  17.19,    0.0,  57.94,    0.0,    0.0],
    [   0.0,    0.0,    0.0,    0.0,  57.94, -17.19],
    [   0.0,    0.0,    0.0,    0.0, -17.19,  39.88]
]

C = StiffnessTensor(matriz, phase_name="Meu Material")

# 2. Extraindo as propriedades direcionais do material
E = C.Young_modulus
G = C.shear_modulus
nu = C.Poisson_ratio

# 3. Gerando as superfícies 3D no padrão Matplotlib (como na sua imagem)

# Plot do Módulo de Young (Superfície única)
#fig_E = E.plot3D(n_phi=360,n_theta=360, backend='matplotlib', cmap='plasma')
#plt.title("Módulo de Young 3D")

# Plot do Módulo de Cisalhamento (Mostra superfícies máxima e mínima)
#figure = plt.figure()
#ax = figure.add_subplot(projection='3d')
# 1. FORÇA o Matplotlib a usar 'Blues' como padrão para TUDO nesta célula

#fig_G = G.plot3D(n_phi=360,n_theta=360,
    #property_name='poisson',
#    isosurface='both',          # Opções: 'max', 'min' ou 'both' (ambas)
    #resolution='180',          # Densidade da malha
    #color_max='red',         # Cor para a superfície máxima
    #color_min='blue',        # Cor para a superfície mínima
    #alpha_max=0.6,           # Transparência do máximo (permite ver o mínimo dentro)
    #alpha_min=0.9,           # Transparência do mínimo
    #scale_mode='absolute',   # Como a escala do raio é calculada
    #grid=True                # Mostrar grade de coordenadas esféricas
#)

# Plot do Coeficiente de Poisson
fig_nu = nu.plot3D(n_phi=150,n_theta=150, backend='plotly', which='minmax', fig='None', opacity=0.90)

fig_nu.data[0].update(
    colorscale=[[0, "#FF0000"], [0.5, "#0011FF"], [1, "#0011FF"]],
    opacity=1.0,
)
fig_nu.data[1].update(
    colorscale=[[0, "#3C8E46"], [1, "#3C8E46"]],#"Viridis",
    opacity=0.6,
            lighting=dict(
        ambient=.8,   # Luz ambiente no máximo: ilumina o objeto 100% por igual
        diffuse=0.99,   # Zera as sombras direcionais das curvas
        specular=0.3,  # Zera o brilho/reflexo (tira o aspecto de plástico/metal)
        roughness=.9, # Deixa a superfície totalmente fosca
        fresnel=0.9    # Tira o contorno brilhante nas bordas
    ),
)

max = math.ceil(nu.max()[0]*10)/10
min = math.floor(nu.min()[0]*10)/10

fig_nu.update_layout(
    scene_camera=dict(
        eye=dict(x=1.28, y=1.28, z=1.28), # Valores maiores que 1.25 afastam a câmera
        center=dict(x=0, y=0, z=-0.15)
    ),
    margin=dict(l=0, r=0, t=0, b=0),
    scene=dict(
        aspectmode='cube',
        xaxis=dict(    
            showbackground=True,
            backgroundcolor="#FFFFFF",
            gridcolor="#D9D9D9",
            gridwidth=4,
            zeroline=False,                      # Limita a quantidade de divisões
            tickfont=dict(
                size=14,
                weight=700,
                color="#535353"
            ),             # Diminui um pouco o tamanho dos números # Afasta a letra 'X' do eixo
            showline=True,
            linecolor="#A7A6A6",  # Cor da linha que une os cantos
            linewidth=4,        # Espessura para dar destaque
            mirror=True,
            showspikes=False,
            autorange=False,
            range=[-max, max],
            title=dict(
                text="ν<sub>x</sub>",
                font=dict(
                    size=14,
                    weight=700,
                ),
            )
        ),
        yaxis=dict(
            showbackground=True,
            backgroundcolor="#FFFFFF",
            gridcolor="#D9D9D9",
            gridwidth=4,
            zeroline=False,
            tickfont=dict(
                size=14,
                weight=700,
                color="#535353"
            ),  
            showline=True,
            linecolor="#A7A6A6",  # Cor da linha que une os cantos
            linewidth=4,        # Espessura para dar destaque
            mirror=True,
            showspikes=False,
            autorange=False,
            range=[-max, max],
            title=dict(
                text="ν<sub>y</sub>",
                font=dict(
                    size=14,
                    weight=700,
                ),
            )
        ),
        zaxis=dict(
            showbackground=True,
            backgroundcolor="#FFFFFF",
            gridcolor="#D9D9D9",
            gridwidth=4,
            zeroline=False,
            tickfont=dict(
                size=14,
                weight=700,
                color="#535353"
            ),  
            showline=True,
            linecolor="#A7A6A6",  # Cor da linha que une os cantos
            linewidth=4,        # Espessura para dar destaque
            mirror=True,
            showspikes=False,
            autorange=False,
            range=[-max, max],
            title=dict(
                text="ν<sub>z</sub>",
                font=dict(
                    size=14,
                    weight=700,
                ),
            )
        )
    )
)

fig_nu.update_traces(
    contours_x=dict(
        show=True,
        usecolormap=False,
        project_z=False, # Não projeta no chão, deixa direto na superfície
        color="#646464",   # Cor das linhas sobre o objeto
        highlightcolor="red",
        project=dict(x=False)
    ),
     contours_y=dict(
        show=True,
        usecolormap=False,
        project_z=False, # Não projeta no chão, deixa direto na superfície
        color="#646464",   # Cor das linhas sobre o objeto
        highlightcolor="red",
        project=dict(y=False)
    ),
     contours_z=dict(
        show=True,
        usecolormap=False,
        project_z=False, # Não projeta no chão, deixa direto na superfície
        color="#646464",   # Cor das linhas sobre o objeto
        highlightcolor="red",
        project=dict(z=False)
    ),
    showscale=False,#
    selector=dict(type='surface')
)

fig_nu.update_traces(
    contours_x=dict(
            #show=True,
            start=0,
            end=5,
            size=0.5,
            usecolormap=True,
            highlight=True,
            highlightwidth=5,
            project_x=True,
            width=16,
    ),
        # Projeta a silhueta/curvas no plano XY (chão do Z)
        #z=dict(show=True, usecolormap=True, project=dict(z=True))
    #),
    selector=dict(type='surface')
)

fig_nu.update_traces(
    showscale=False,#
)

fig_nu.write_image("fig.png", width=800, height=600, scale=3)
#fig_nu.show()

KeyboardInterrupt: 

In [35]:
import math
from elasticipy.tensors.elasticity import StiffnessTensor
import matplotlib.pyplot as plt
import numpy as np
import plotly.graph_objects as go

# 1. Criando o tensor
matriz = [
    [87.64, 6.99, 11.91, -17.19, 0.0, 0.0],
    [6.99, 87.64, 11.91, 17.19, 0.0, 0.0],
    [11.91, 11.91, 107.20, 0.0, 0.0, 0.0],
    [-17.19, 17.19, 0.0, 57.94, 0.0, 0.0],
    [0.0, 0.0, 0.0, 0.0, 57.94, -17.19],
    [0.0, 0.0, 0.0, 0.0, -17.19, 39.88],
]

#matriz = [
#    [83.504, 50.229, 50.229, 0.0, 0.0, 0.0],
#    [50.229, 83.504, 50.229, 0.0, 0.0, 0.0],
#    [50.229, 50.229, 83.504, 0.0, 0.0, 0.0],
#    [0.0, 0.0, 0.0, 19.636, 0.0, 0.0],
#    [0.0, 0.0, 0.0, 0.0, 19.636, 0.0],
#    [0.0, 0.0, 0.0, 0.0, 0.0, 19.636],
#]

C = StiffnessTensor(matriz, phase_name="Meu Material")
nu = C.linear_compressibility#Poisson_ratio

print(nu)

nu = nu*1000

# 2. Gerando o gráfico inicial do Coeficiente de Poisson
fig_nu = nu.plot3D(
    n_phi=150, n_theta=150, backend="plotly", opacity=0.90
)

print()

max_val = math.ceil(nu.max()[0]) #* 100) / 100
min_val = math.floor(nu.min()[0]) #* 100) / 100

# 3. Customização das superfícies do elasticipy
fig_nu.data[0].update(
    cmin=-max_val,
    cmax=max_val,
    colorscale=[[0, "#FF0000"], [0.5, "#0011FF"], [1, "#0011FF"]],
    opacity=.6,
)
#fig_nu.data[1].update(
#    colorscale=[[0, "#3C8E46"], [1, "#3C8E46"]],
#    opacity=0.6,
#    lighting=dict(
#        ambient=0.8, diffuse=0.99, specular=0.3, roughness=0.9, fresnel=0.9
#    ),
#)



# 4. Configuração da cena e dos eixos
fig_nu.update_layout(
    scene_camera=dict(
        eye=dict(x=1.28, y=1.28, z=1.28), center=dict(x=0, y=0, z=-0.15)
    ),
    margin=dict(l=0, r=0, t=0, b=0),
    scene=dict(
        aspectmode="cube",
        xaxis=dict(
            showbackground=True,
            backgroundcolor="#FFFFFF",
            gridcolor="#D9D9D9",
            gridwidth=4,
            zeroline=False,
            tickfont=dict(size=14, weight=700, color="#535353"),
            showline=True,
            linecolor="#A7A6A6",
            linewidth=4,
            mirror=True,
            showspikes=False,
            autorange=False,
            range=[-max_val, max_val],
            title=dict(text="ν<sub>x</sub>", font=dict(size=14, weight=700)),
        ),
        yaxis=dict(
            showbackground=True,
            backgroundcolor="#FFFFFF",
            gridcolor="#D9D9D9",
            gridwidth=4,
            zeroline=False,
            tickfont=dict(size=14, weight=700, color="#535353"),
            showline=True,
            linecolor="#A7A6A6",
            linewidth=4,
            mirror=True,
            showspikes=False,
            autorange=False,
            range=[-max_val, max_val],
            title=dict(text="ν<sub>y</sub>", font=dict(size=14, weight=700)),
        ),
        zaxis=dict(
            showbackground=True,
            backgroundcolor="#FFFFFF",
            gridcolor="#D9D9D9",
            gridwidth=4,
            zeroline=False,
            tickfont=dict(size=14, weight=700, color="#535353"),
            showline=True,
            linecolor="#A7A6A6",
            linewidth=4,
            mirror=True,
            showspikes=False,
            autorange=False,
            range=[-max_val, max_val],
            title=dict(text="ν<sub>z</sub>", font=dict(size=14, weight=700)),
        ),
    ),
)

# --- FUNÇÃO DE INTERPOLAÇÃO PARA DIRETAMENTE DA MALHA PARAMÉTRICA ---
def interp2d_points(matrix, rows, cols):
    r0 = np.floor(rows).astype(int)
    r1 = np.minimum(r0 + 1, matrix.shape[0] - 1)
    c0 = np.floor(cols).astype(int)
    c1 = np.minimum(c0 + 1, matrix.shape[1] - 1)
    dr, dc = rows - r0, cols - c0
    return (
        matrix[r0, c0] * (1 - dr) * (1 - dc)
        + matrix[r1, c0] * dr * (1 - dc)
        + matrix[r0, c1] * (1 - dr) * dc
        + matrix[r1, c1] * dr * dc
    )

# 5. GERADOR UNIFICADO DE PROJEÇÕES (CORREÇÃO DE PATHS E PÉTALAS CORTADAS)
contour_vals = [1e-5] # Deslocamento microscópico para evitar buracos na malha zero exata

base_val = -max_val
epsilon = (2 * max_val) * 0.01  
line_width = 4  

planes_data = {
    'YZ': {'red': {'x':[], 'y':[], 'z':[]}, 'blue': {'x':[], 'y':[], 'z':[]}, 'green': {'x':[], 'y':[], 'z':[]}},
    'ZX': {'red': {'x':[], 'y':[], 'z':[]}, 'blue': {'x':[], 'y':[], 'z':[]}, 'green': {'x':[], 'y':[], 'z':[]}},
    'XY': {'red': {'x':[], 'y':[], 'z':[]}, 'blue': {'x':[], 'y':[], 'z':[]}, 'green': {'x':[], 'y':[], 'z':[]}}
}

fig_mock, ax_mock = plt.subplots()

for trace in list(fig_nu.data):
    print(trace)
    if trace.type == "surface":
        X_data = np.array(trace.x)
        Y_data = np.array(trace.y)
        Z_data = np.array(trace.z)
        
        has_surfacecolor = hasattr(trace, 'surfacecolor') and trace.surfacecolor is not None
        color_data = np.array(trace.surfacecolor) if has_surfacecolor else Z_data

        is_green_surface = False
        if hasattr(trace, 'colorscale') and trace.colorscale is not None:
            scale_str = str(trace.colorscale).lower()
            if 'green' in scale_str or '#3c8e46' in scale_str:
                is_green_surface = True
        
        if not is_green_surface and np.nanmin(color_data) >= 0 and np.nanmax(color_data) > 0.4:
            is_green_surface = True

        configs = [
            ('YZ', X_data, Y_data, Z_data, 'x'), 
            ('ZX', Y_data, X_data, Z_data, 'y'), 
            ('XY', Z_data, X_data, Y_data, 'z')  
        ]

        for plane_name, slice_data, interp1, interp2, proj_axis in configs:
            
            min_val_slice = np.nanmin(slice_data)
            max_val_slice = np.nanmax(slice_data)
            
            if min_val_slice > 1e-4 or max_val_slice < -1e-4:
                continue
                
            try:
                cs = ax_mock.contour(slice_data, levels=contour_vals)
            except ValueError:
                continue
            
            for path in cs.get_paths():
                v = path.vertices
                c = path.codes 

                segments = []
                if c is None:
                    segments.append(v)
                else:
                    cur_seg = []
                    for idx in range(len(v)):
                        code = c[idx]
                        if code == 1: # MOVETO (Começa nova pétala)
                            if cur_seg:
                                segments.append(np.array(cur_seg))
                            cur_seg = [v[idx]]
                        elif code == 79: # CLOSEPOLY (Fecha a pétala sem usar o (0,0) falso do matplotlib)
                            if cur_seg:
                                cur_seg.append(cur_seg[0]) # Liga o último ponto ao primeiro perfeitamente
                                segments.append(np.array(cur_seg))
                            cur_seg = []
                        else: # LINETO (Continua desenhando)
                            cur_seg.append(v[idx])
                    if cur_seg:
                        segments.append(np.array(cur_seg))

                for seg in segments:
                    if len(seg) < 2:
                        continue
                        
                    cols, rows = seg[:, 0], seg[:, 1]

                    # Mantém a interpolação perfeita que gera as curvas suaves
                    p1_points = interp2d_points(interp1, rows, cols)
                    p2_points = interp2d_points(interp2, rows, cols)
                    line_color_values = interp2d_points(color_data, rows, cols)

                    def add_segment(color_category, arr1, arr2, offset_layer):
                        n_pts = len(arr1)
                        if proj_axis == 'x':
                            xs = [offset_layer] * n_pts
                            ys = list(arr1)
                            zs = list(arr2)
                        elif proj_axis == 'y':
                            ys = [offset_layer] * n_pts
                            xs = list(arr1)
                            zs = list(arr2)
                        else: 
                            zs = [offset_layer] * n_pts
                            xs = list(arr1)
                            ys = list(arr2)
                            
                        planes_data[plane_name][color_category]['x'].extend(xs + [None])
                        planes_data[plane_name][color_category]['y'].extend(ys + [None])
                        planes_data[plane_name][color_category]['z'].extend(zs + [None])

                    if is_green_surface:
                        add_segment('green', p1_points, p2_points, base_val + epsilon/2.0)
                    else:
                        for i in range(len(p1_points) - 1):
                            v1, v2 = line_color_values[i], line_color_values[i+1]
                            
                            if np.isnan(v1) or np.isnan(v2):
                                continue
                                
                            p1_a, p1_b = p1_points[i], p1_points[i+1]
                            p2_a, p2_b = p2_points[i], p2_points[i+1]
                            
                            offset_red = base_val + epsilon/2.6
                            offset_blue = base_val +  epsilon/2.3
                            
                            if v1 < 0 and v2 < 0:
                                add_segment('red', [p1_a, p1_b], [p2_a, p2_b], offset_red)
                            elif v1 >= 0 and v2 >= 0:
                                add_segment('blue', [p1_a, p1_b], [p2_a, p2_b], offset_blue)
                            else:
                                t = (0.0 - v1) / (v2 - v1) if (v2 - v1) != 0 else 0.5
                                p1_zero = p1_a + t * (p1_b - p1_a)
                                p2_zero = p2_a + t * (p2_b - p2_a)
                                
                                if v1 < 0:
                                    add_segment('red', [p1_a, p1_zero], [p2_a, p2_zero], offset_red)
                                    add_segment('blue', [p1_zero, p1_b], [p2_zero, p2_b], offset_blue)
                                else:
                                    add_segment('blue', [p1_a, p1_zero], [p2_a, p2_zero], offset_blue)
                                    add_segment('red', [p1_zero, p1_b], [p2_zero, p2_b], offset_red)

plt.close(fig_mock)

fig_nu.update_traces(
    contours_x=dict(
        show=True,
        usecolormap=False,
        project_z=False, # Não projeta no chão, deixa direto na superfície
        color="#646464",   # Cor das linhas sobre o objeto
        highlightcolor="red",
        project=dict(x=False)
    ),
     contours_y=dict(
        show=True,
        usecolormap=False,
        project_z=False, # Não projeta no chão, deixa direto na superfície
        color="#646464",   # Cor das linhas sobre o objeto
        highlightcolor="red",
        project=dict(y=False)
    ),
     contours_z=dict(
        show=True,
        usecolormap=False,
        project_z=False, # Não projeta no chão, deixa direto na superfície
        color="#646464",   # Cor das linhas sobre o objeto
        highlightcolor="red",
        project=dict(z=False)
    ),
    showscale=False,#
    selector=dict(type='surface')
)

# 6. ADICIONAR PROJEÇÕES AO GRÁFICO ORIGINAL
for plane_name, colors in planes_data.items():
    for color_category, coords in colors.items():
        if coords['x']:
            # Define a cor real da linha baseada na categoria
            if color_category == 'green':
                line_color = '#3c8e46'
            elif color_category == 'red':
                line_color = 'red'
            else:
                line_color = 'blue'
                
            fig_nu.add_trace(go.Scatter3d(
                x=coords['x'], y=coords['y'], z=coords['z'],
                mode='lines',
                line=dict(color=line_color, width=line_width),
                name=f"Proj {plane_name} ({color_category})",
                showlegend=False
            ))
# 7. Exportação final limpa
fig_nu.write_image("fig.png", width=800, height=600, scale=3)

Spherical function
Min=0.0071810756133694285, Max=0.009663673142182926

Surface({
    'cmax': 10,
    'cmin': -10,
    'colorscale': [[0, '#FF0000'], [0.5, '#3C8E46'], [1, '#3C8E46']],
    'customdata': array([[[ 7.18107561e+00,  0.00000000e+00,  0.00000000e+00],
                          [ 7.18107561e+00,  0.00000000e+00,  0.00000000e+00],
                          [ 7.18107561e+00,  0.00000000e+00,  0.00000000e+00],
                          ...,
                          [ 7.18107561e+00,  0.00000000e+00, -0.00000000e+00],
                          [ 7.18107561e+00,  0.00000000e+00, -0.00000000e+00],
                          [ 7.18107561e+00,  0.00000000e+00, -0.00000000e+00]],
                  
                         [[ 7.18217911e+00,  1.20805369e+00,  0.00000000e+00],
                          [ 7.18217911e+00,  1.20805369e+00,  2.41610738e+00],
                          [ 7.18217911e+00,  1.20805369e+00,  4.83221477e+00],
                          ...,
                      

In [12]:
import matplotlib.pyplot as plt
from ase.eos import EquationOfState
from ase.units import kJ

en = [-268.4852,-268.4626,-268.4525,-268.3875,-268.2049]
vol = [0.263338E+03,0.271482E+03,0.279792E+03,0.288270E+03,0.296917E+03]

plt.close()

birch_murnaghan = EquationOfState(vol, en, eos='birchmurnaghan')
B = birch_murnaghan.fit()
birch_murnaghan.plot()

#Ge = 97.56 GPa
#Ge free = 95.32 GPa


RuntimeError: Optimal parameters not found: Number of calls to function has reached maxfev = 1000.

In [65]:
from ase.dft.kpoints import get_special_points, bandpath
from ase.geometry import Cell
import numpy as np

# Uma lista de listas contendo os vetores [v1, v2, v3]
lista_vetores = [
    [5.2, 0.0, 0.0],  # Vetor a
    [0.0, 5.2, 0.0],  # Vetor b
    [0.0, 0.0, 5.2],  # Vetor c
]

# Cria o objeto Cell diretamente
cell = Cell(lista_vetores)

#cell = Cell.fromcellpar([3,3,0,90,90,90])
latt = cell.get_bravais_lattice() 
print(list(latt.get_special_points()))
print(latt.bandpath().special_points)
print(latt.name)

['G', 'M', 'R', 'X']
{'G': array([0., 0., 0.]), 'M': array([0.5, 0.5, 0. ]), 'R': array([0.5, 0.5, 0.5]), 'X': array([0. , 0.5, 0. ])}
CUB
